In [1]:
import geopandas as gpd
import pygris
import requests
import time
import pandas as pd

osm_landuse = "OpenStreetMap/gis_osm_landuse_a_free_1.shp"  
output = "urban_farms_with_addresses.csv"
counties = ["Dallas", "Harris", "Tarrant", "Travis", "Bexar"]

counties = pygris.counties(state="TX", year=2025)
counties = counties[counties["NAME"].isin(counties)].copy()

landuse = gpd.read_file(osm_landuse)

# Keep all allotments + only named farmland
farms = landuse[
    (landuse["fclass"] == "allotments") |
    (
        (landuse["fclass"] == "farmland") &
        (landuse["name"].notna()) &
        (landuse["name"] != "")
    )
].copy()

if farms.crs != counties.crs:
    counties = counties.to_crs(farms.crs)

farms = gpd.sjoin(
    farms,
    counties[["NAME", "COUNTYFP", "geometry"]],
    how="inner",
    predicate="intersects"
).rename(columns={"NAME": "county"})


farms_projected = farms.to_crs("EPSG:3857")
centroids = farms_projected.geometry.centroid.to_crs("EPSG:4326")
farms["lat"] = centroids.y
farms["lon"] = centroids.x

def reverse_geocode(lat, lon):
    try:
        url = "https://nominatim.openstreetmap.org/reverse"
        params = {"lat": lat, "lon": lon, "format": "json"}
        headers = {"User-Agent": "urban-farm-research-tx"}
        response = requests.get(url, params=params, headers=headers, timeout=10)
        if response.status_code == 200:
            data = response.json()
            address = data.get("address", {})
            return (
                data.get("display_name", ""),
                address.get("house_number", ""),
                address.get("road", ""),
                address.get("city", address.get("town", address.get("suburb", ""))),
                address.get("postcode", "")
            )
    except Exception as e:
        print(f"  Error: {e}")
    return "", "", "", "", ""

full_addresses, house_numbers, streets, cities, zipcodes = [], [], [], [], []

for i, (_, row) in enumerate(farms.iterrows()):
    print(f"  {i+1}/{len(farms)} — osm_id {row['osm_id']}", end="\r")
    full, house, street, city, zipcode = reverse_geocode(row["lat"], row["lon"])
    full_addresses.append(full)
    house_numbers.append(house)
    streets.append(street)
    cities.append(city)
    zipcodes.append(zipcode)
    time.sleep(1)

farms["full_address"] = full_addresses
farms["house_number"] = house_numbers
farms["street"] = streets
farms["city"] = cities
farms["zipcode"] = zipcodes

export_cols = ["osm_id", "fclass", "name", "county", "COUNTYFP",
               "lat", "lon", "house_number", "street", "city",
               "zipcode", "full_address"]

farms[export_cols].to_csv(output, index=False)

Using FIPS code '48' for input 'TX'
